# Module 03-2: 3노드 그래프 + Mermaid 시각화 + Stream 실행

## 학습 목표

1. **FakeLLM 활용**: `common.fake_llm.FakeLLM`을 노드 내에서 사용하는 패턴을 익힌다
2. **3노드 분석 파이프라인**: analyze → summarize → format 흐름을 구현한다
3. **Mermaid 시각화**: `get_graph().draw_mermaid()`로 그래프 구조를 시각화한다
4. **Stream 실행**: `stream()` 메서드로 노드별 중간 상태를 확인한다
5. **invoke vs stream 비교**: 두 실행 방식의 차이를 이해한다

---
> 참고 문서: `docs/part2-first-agent/03-jupyter-dev-environment.md`

## 개념 요약

### invoke vs stream

```python
# invoke: 최종 결과만 반환
result = app.invoke(initial_state)
print(result)  # 최종 상태 딕셔너리

# stream: 노드 실행마다 중간 상태 반환
for chunk in app.stream(initial_state):
    print(chunk)  # {"노드명": {변경된_필드: 값}}
```

### Mermaid 시각화

```python
# 텍스트 형식으로 출력
mermaid_code = app.get_graph().draw_mermaid()
print(mermaid_code)

# 주피터에서 PNG 렌더링 (선택적)
from IPython.display import Image
Image(app.get_graph().draw_mermaid_png())
```

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from common.fake_llm import FakeLLM

print("환경 설정 완료")

## 실습 1: TextAnalysisState 정의 및 FakeLLM 초기화

텍스트 분석 그래프에 사용할 상태와 FakeLLM을 설정하세요.

**TextAnalysisState 필드:**
- `text`: 분석할 원본 텍스트
- `analysis`: 분석 결과
- `summary`: 요약 결과
- `final_report`: 최종 보고서

**FakeLLM 응답 패턴:**
- `"분석"` → `"키워드 분석 완료: 기술, AI, 에이전트 관련 내용"`
- `"요약"` → `"핵심 요약: LangGraph로 에이전트를 구축하는 방법"`

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: TypedDict로 TextAnalysisState를 정의합니다
# 힌트 2: FakeLLM(responses={...})으로 패턴-응답을 설정합니다
# 힌트 3:
# class TextAnalysisState(TypedDict):
#     text: str
#     analysis: str
#     summary: str
#     final_report: str
#
# llm = FakeLLM(responses={
#     "분석": "키워드 분석 완료: 기술, AI, 에이전트 관련 내용",
#     "요약": "핵심 요약: LangGraph로 에이전트를 구축하는 방법"
# })

# TextAnalysisState를 정의하고 FakeLLM을 초기화하세요

print("TextAnalysisState와 FakeLLM 설정 완료")

In [ ]:
# 검증 셀
assert 'TextAnalysisState' in dir(), "TextAnalysisState가 정의되어야 합니다"
assert 'llm' in dir(), "llm 변수가 정의되어야 합니다"
assert isinstance(llm, FakeLLM), "llm은 FakeLLM 인스턴스여야 합니다"
assert len(llm.responses) >= 2, "최소 2개의 패턴-응답이 필요합니다"
print("✅ 실습 1 완료!")

## 실습 2: 3개 노드 함수 구현

FakeLLM을 사용하는 3개의 노드를 구현하세요:

1. `analyze(state)`: `f"다음 텍스트를 분석해주세요: {text}"` 프롬프트로 LLM 호출 → `analysis`
2. `summarize(state)`: `f"다음 분석 결과를 요약해주세요: {analysis}"` 프롬프트로 LLM 호출 → `summary`
3. `format_report(state)`: 텍스트, 분석, 요약을 합쳐 최종 보고서 생성 → `final_report`

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: llm.invoke("프롬프트").content로 응답을 가져옵니다
# 힌트 2: format_report는 LLM을 호출하지 않고 상태 값을 조합합니다
# 힌트 3:
# def analyze(state: TextAnalysisState) -> dict:
#     prompt = f"다음 텍스트를 분석해주세요: {state['text']}"
#     response = llm.invoke(prompt)
#     return {"analysis": response.content}
#
# def format_report(state: TextAnalysisState) -> dict:
#     report = (
#         f"=== 텍스트 분석 보고서 ===\n"
#         f"원본: {state['text']}\n"
#         f"분석: {state['analysis']}\n"
#         f"요약: {state['summary']}"
#     )
#     return {"final_report": report}

# analyze, summarize, format_report 함수를 구현하세요

print("노드 함수 구현 완료")

In [ ]:
# 검증 셀
test_state = {"text": "LangGraph 분석 테스트", "analysis": "", "summary": "", "final_report": ""}

r_analyze = analyze(test_state)
assert "analysis" in r_analyze, "analyze는 'analysis' 키를 반환해야 합니다"
assert len(r_analyze["analysis"]) > 0, "analysis가 비어있지 않아야 합니다"

test_state["analysis"] = r_analyze["analysis"]
r_summarize = summarize(test_state)
assert "summary" in r_summarize, "summarize는 'summary' 키를 반환해야 합니다"

test_state["summary"] = r_summarize["summary"]
r_format = format_report(test_state)
assert "final_report" in r_format, "format_report는 'final_report' 키를 반환해야 합니다"
assert "분석" in r_format["final_report"] or "원본" in r_format["final_report"], \
    "final_report에 분석 내용이 포함되어야 합니다"
print("✅ 실습 2 완료!")

## 실습 3: 그래프 조립 및 Mermaid 시각화

그래프를 조립하고 Mermaid 형식으로 시각화하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: START → analyze → summarize → format_report → END 순서
# 힌트 2: app.get_graph().draw_mermaid()로 Mermaid 코드를 출력합니다
# 힌트 3:
# graph = StateGraph(TextAnalysisState)
# graph.add_node("analyze", analyze)
# ... (나머지 노드와 엣지)
# app = graph.compile()
# print(app.get_graph().draw_mermaid())

# 그래프를 조립하고 Mermaid로 시각화하세요

print("그래프 조립 및 시각화 완료")

In [ ]:
# 검증 셀
assert 'app' in dir(), "app 변수가 정의되어야 합니다"
mermaid_code = app.get_graph().draw_mermaid()
assert "analyze" in mermaid_code, "Mermaid 코드에 analyze 노드가 있어야 합니다"
assert "summarize" in mermaid_code, "Mermaid 코드에 summarize 노드가 있어야 합니다"
assert "format_report" in mermaid_code, "Mermaid 코드에 format_report 노드가 있어야 합니다"
print("Mermaid 코드:")
print(mermaid_code)
print("✅ 실습 3 완료!")

## 실습 4: Stream 실행으로 중간 상태 확인

`stream()` 메서드를 사용하여 각 노드 실행 시 중간 상태를 확인하세요.

In [ ]:
# TODO: 여기에 코드를 작성하세요

# 힌트 1: app.stream(초기상태)는 이터레이터를 반환합니다
# 힌트 2: 각 청크는 {"노드명": {변경된 필드: 값}} 형식입니다
# 힌트 3:
# initial_state = {"text": "LangGraph를 배우는 중입니다", "analysis": "", "summary": "", "final_report": ""}
# for chunk in app.stream(initial_state):
#     node_name = list(chunk.keys())[0]
#     print(f"[{node_name}] 실행됨")
#     print(chunk)

# stream으로 실행하고 각 노드의 중간 상태를 출력하세요

In [ ]:
# 검증 셀 - invoke와 stream 결과 비교
initial = {"text": "LangGraph 검증 테스트", "analysis": "", "summary": "", "final_report": ""}

# invoke 결과
invoke_result = app.invoke(initial)

# stream 결과 수집
stream_chunks = list(app.stream(initial))
assert len(stream_chunks) == 3, f"stream은 3개의 청크를 반환해야 합니다 (노드 수). 실제: {len(stream_chunks)}"

# 노드 이름 확인
node_names = [list(chunk.keys())[0] for chunk in stream_chunks]
assert "analyze" in node_names, "stream에 analyze 청크가 있어야 합니다"
assert "summarize" in node_names, "stream에 summarize 청크가 있어야 합니다"
assert "format_report" in node_names, "stream에 format_report 청크가 있어야 합니다"

print(f"invoke 최종 보고서:\n{invoke_result['final_report']}")
print(f"\nstream 청크 수: {len(stream_chunks)}")
print("✅ 실습 4 완료!")

## 정리

### 오늘 배운 내용
- `FakeLLM`을 노드 내에서 사용하여 **API 없이 LLM 호출 시뮬레이션**
- `get_graph().draw_mermaid()`로 **그래프 구조 시각화**
- `stream()` vs `invoke()`: 중간 상태 확인 vs 최종 결과만 반환

### 다음 모듈 안내
**Module 04: 첫 에이전트** - 에러 처리와 조건부 분기가 포함된 실용적인 에이전트를 구축합니다.